# Function Calling with Gemini 2.5

In [1]:
from google import genai
client = genai.Client()

In [2]:
import requests,json
def get_current_weather(city:str)->dict:
    """ can be used to get/fetch current weather information for a city name
    """
    api_key = "6a8b0ac166a37e2b7a38e64416b3c3fe"

    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}"
    response = requests.get(url)
    response = response.content.decode()
    response = json.loads(response)
    output = {"City Name":city,"weather":response["weather"][0]['description'],
              "temperature":response['main']['temp'],
              "unit":"kelvin"}

    return output

In [3]:
get_current_weather("mumbai")

{'City Name': 'mumbai',
 'weather': 'scattered clouds',
 'temperature': 300.07,
 'unit': 'kelvin'}

In [4]:
get_current_weather("delhi")

{'City Name': 'delhi',
 'weather': 'few clouds',
 'temperature': 308.09,
 'unit': 'kelvin'}

### Tool Integration with LLM

- metadata for the tool
    - metdata needs to include 3 compoents
        - name of function
        - description of function
        - arguments: args and its description
    - OPEN API Format to define metadata

In [21]:
tools_def = [{
    "name":"get_current_weather",
    "description":"this function is used to get/fetch current weather information for any given city",
    "parameters":{"type":"object",
                  "properties":{"city":{"type":"string","description":"name of any location/city e.g. mumbai, new york"}},
                  "required":["city"],},
                  
},]


In [47]:
from google.genai import types
import json
tools = types.Tool(function_declarations=tools_def)
config = types.GenerateContentConfig(tools=[tools],
                                     automatic_function_calling=types.FunctionCallingConfig(mode='AUTO')) #NONE, ANY

tool_map = {"get_current_weather":get_current_weather}


In [57]:
def generate_response(prompt:str):
    contents = [types.Content(role='user',parts=[types.Part(text=prompt)])]

    first_response=client.models.generate_content(model='gemini-2.0-flash',
                                                  contents=contents,
                                                  config=config
                                                  )
    #print(first_response)
    if first_response.candidates[0].content.parts[0].function_call:
        # do something
        print("LLM decided to make a function call,", first_response.candidates[0].content.parts[0].function_call)
        tool_call = first_response.candidates[0].content.parts[0].function_call
        contents.append(first_response.candidates[0].content)
        
        tool_name = tool_call.name
        tool_args = tool_call.args
        function_to_ex = tool_map[tool_name]
        tool_output = function_to_ex(**tool_args)

        tool_resp = types.Part.from_function_response(name=tool_name,response={"result":tool_output})

        contents.append(types.Content(role='user',parts=[tool_resp]))

        second_response = client.models.generate_content(model='gemini-2.0-flash',
                                                         contents=contents)
        return second_response.text


        
    else:
        return first_response.text

In [58]:
generate_response("define quantum computing?")

'Quantum computing is a type of computing that uses quantum mechanics principles to solve complex problems that classical computers cannot handle efficiently.\n'

In [59]:
generate_response("what is the current weather in delhi?")

LLM decided to make a function call, id=None args={'city': 'delhi'} name='get_current_weather'


'The current weather in Delhi is few clouds and the temperature is 308.09 Kelvin.\n'